# 2-D rate maps: session time × reward time (PFC)

One rate map per neuron over the two nested time axes, pooled across the sessions
of its recday. Module: [`time_manifold.py`](time_manifold.py). Companion doc:
[`TIME_MANIFOLD.md`](TIME_MANIFOLD.md).

**Two variants**, both axes changing together:

| variant | slow axis | fast axis |
|---|---|---|
| `normalised` | fraction of session | fraction of the goal→goal transition |
| `absolute` | elapsed time in session | elapsed seconds since last reward |

Comparing them is the "bin the fast axis both ways" logic applied to both axes at
once: a field that is sharp in the normalised map and smeared in the absolute one
is tracking phase, not a clock.

## Read this before reading the maps

PFC fires at a **median of 0.50 Hz** (p10 0.10, p90 1.62) — about 5× LEC, so these maps are the cleaner of the two regions. On a 20×15 grid a cell holds roughly 7.5–14 s of data, so a **median
LEC neuron contributes under one spike per bin** (a median PFC neuron ~10).

This pass is **deliberately descriptive**: no reliability statistic, no null, no
significance test. Nothing here separates a real field from Poisson noise. Many
individual maps — most LEC ones — will be noise, and they will still look like
something once smoothed. Treat this as a first look at the data, not as evidence.

Two further caveats:

- **Sessions are different tasks.** `get_sessions_for_glm` dedups to one session
  per unique task, so pooling averages over the place↔state remapping that
  `remapping_rotation_analysis` measures. If a neuron's time tuning is
  task-specific it washes out here. `neuron_ratemaps(..., sessions=[s])` gives a
  per-session version.
- The **absolute** variant caps its slow axis at the *shortest* session, so every
  bin is supported by every session. Raising `config.T_range_sec` re-introduces a
  tail built from the long sessions only.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os, pickle
from importlib import reload
import time_manifold as tm; reload(tm)
import glm_analysis_v2 as glm
glm.apply_gridmaze_style()

GRID = dict(n_T=20, n_tau=15)   # see the occupancy budget below before changing
SMOOTH_BINS = 1.0
SAVE_FIGS = True

In [ ]:
# PFC data -- same loader the other PFC notebooks use
import os, numpy as np, pickle
from glm_analysis_v2 import build_data_dic_from_pfc
DATA_FOLDER = "/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data"
META = os.path.join(DATA_FOLDER, "MetaData")
mouse_recdays = [str(mr) for mr in np.load(os.path.join(META, "combined_ABCDonly_days.npy"))]
data_dic = build_data_dic_from_pfc(DATA_FOLDER, mouse_recdays)
mouse_recdays = sorted(data_dic.keys())
REGION = "PFC"
SAVE_DIR = "../glm_outputs/PFC_time_ratemaps"
print(f"{len(mouse_recdays)} PFC recdays")

## 1. Build the substrate

`build_time_tables` is the same substrate every other time-manifold analysis uses
(same session dedup, same `Locs <= 21` node filter, same transition mask), plus
`T_sec` — elapsed seconds since session start.

In [ ]:
cfg = tm.TimeManifoldConfig(**GRID)
tables = tm.build_time_tables(mouse_recdays, data_dic, cfg)
n_neurons = sum(t["FR"].shape[0] for t in tables.values())
print(f"\n{len(tables)} recdays, {n_neurons} neurons")

## 2. The rate and occupancy budget for *this* run

Printed rather than quoted, so the sparsity is visible for the grid actually
chosen. `spikes/cell` is the number that decides whether a single map means
anything: below ~1 the map is dominated by Poisson noise.

In [ ]:
rows = []
for rd, t in tables.items():
    hz = t["FR"].mean(1) / t["bin_seconds"]
    rm = tm.neuron_ratemaps(t, cfg, variant="absolute", smooth_bins=0.0)
    s_per_cell = float(np.median(rm["seconds"]))
    rows.append((rd, t["FR"].shape[0], np.median(hz), s_per_cell,
                 s_per_cell * np.median(hz), int((rm["occupancy"] == 0).sum())))

print(f"{'recday':26s} {'N':>4s} {'med Hz':>7s} {'s/cell':>7s} "
      f"{'spikes/cell':>12s} {'empty cells':>12s}")
for r in rows:
    print(f"{r[0]:26s} {r[1]:4d} {r[2]:7.2f} {r[3]:7.1f} {r[4]:12.2f} {r[5]:12d}")

med = np.median([r[4] for r in rows])
print(f"\nmedian neuron gets {med:.2f} spikes per (T, tau) cell")
if med < 1.0:
    print("  -> below 1 spike/cell: individual maps are mostly Poisson noise.")

## 3. Build both variants

For each neuron we keep three things: the **raw** map, the **smoothed** map, and
the 2-D **autocorrelogram** of the smoothed map. All cached to `.npz` so
re-plotting never re-reads the data dictionary.

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
ratemaps = {}
for variant in ("normalised", "absolute"):
    print(f"--- {variant} ---")
    out = tm.build_ratemaps_all(tables, cfg, variant=variant,
                                smooth_bins=SMOOTH_BINS, autocorr=True)
    ratemaps[variant] = out
    path = os.path.join(SAVE_DIR, f"ratemaps_{variant}.npz")
    np.savez_compressed(
        path, maps=out["maps"], maps_raw=out["maps_raw"],
        autocorr=out["autocorr"], autocorr_raw=out["autocorr_raw"],
        recday=out["meta"]["recday"].values, neuron=out["meta"]["neuron"].values,
        mean_hz=out["meta"]["mean_hz"].values, peak_hz=out["meta"]["peak_hz"].values,
        **{k: out["axes"][k] for k in
           ("T_centres", "fast_centres", "T_label", "fast_label", "variant",
            "smooth_bins", "T_window", "fast_window", "T_bin", "fast_bin")})
    print(f"  {out['maps'].shape[0]} neurons, grid "
          f"{out['maps'].shape[1]}x{out['maps'].shape[2]}, "
          f"autocorr {out['autocorr'].shape[1]}x{out['autocorr'].shape[2]}")
    print(f"  -> {path}\n")

### 3b. Reloading from the cache

`load_ratemaps_npz` reconstructs the `axes` dict with everything the plotters
need — including `smooth_bins` and the per-axis bin widths, without which the 2σ
kernel ellipse cannot be placed in data coordinates.

In [ ]:
def load_ratemaps_npz(path):
    """npz -> the same (maps, maps_raw, autocorr, autocorr_raw, meta, axes) dict
    that `build_ratemaps_all` returns."""
    import pandas as pd
    d = np.load(path, allow_pickle=True)
    axes = {k: (d[k].item() if d[k].ndim == 0 else d[k]) for k in
            ("T_centres", "fast_centres", "T_label", "fast_label", "variant",
             "smooth_bins", "T_window", "fast_window", "T_bin", "fast_bin")}
    meta = pd.DataFrame(dict(recday=d["recday"], neuron=d["neuron"],
                             mean_hz=d["mean_hz"], peak_hz=d["peak_hz"]))
    return dict(maps=d["maps"], maps_raw=d["maps_raw"], autocorr=d["autocorr"],
                autocorr_raw=d["autocorr_raw"], meta=meta, axes=axes)

# e.g. ratemaps = {v: load_ratemaps_npz(os.path.join(SAVE_DIR, f"ratemaps_{v}.npz"))
#                  for v in ("normalised", "absolute")}
print("reload helper defined")

## 4. Occupancy and where the peaks fall

Read the occupancy panel first — it says which parts of the plane are supported
by data at all. The long-leg corner is always thin.

In [ ]:
for variant in ("normalised", "absolute"):
    rm = tm.neuron_ratemaps(list(tables.values())[0], cfg, variant=variant,
                            smooth_bins=SMOOTH_BINS)
    fig = tm.plot_ratemap_summary(rm)
    fig.suptitle(f"{REGION} · {variant} · {list(tables)[0]}", fontsize=8, y=1.02)
    if SAVE_FIGS:
        fig.savefig(os.path.join(SAVE_DIR, f"summary_{variant}.pdf"),
                    bbox_inches="tight")
    plt.show()

## 5. Every neuron: raw | smoothed | autocorr(raw) | autocorr(smoothed)

One row per neuron, four panels.

**Raw sits beside smoothed on purpose.** At these rates the raw map is close to
unreadable and the smoothed one is the only interpretable version — but smoothing
is also what can invent a field. Each map therefore sits beside *its own*
autocorrelogram, so the kernel's contribution is visible by comparison rather than
taken on trust. The raw autocorrelogram will often be close to a delta at zero lag
plus noise; at 0.75 spikes per bin that is the honest picture.

**The dashed ellipse** on the smoothed autocorrelogram marks 2σ of the smoothing
kernel. A Gaussian of width σ has an autocorrelation that is itself Gaussian with
std √2σ (∝ exp(−d²/4σ²)), so at 2σ it has fallen to e⁻¹ = 0.37 of the centre.
Structure *inside* the ellipse is substantially the kernel; only structure
*outside* it can be the neuron. It is an ellipse rather than a circle because the
two axes have different bin widths — at the 20×15 `absolute` default, 2σ is
1.70 min on the slow axis and 3.97 s on the fast one.

Axes: `viridis` for the maps (non-negative rate), `RdBu_r` centred at zero for the
autocorrelograms (a signed correlation). Ticks are on the bottom row only, since
every row shares the same axes; the column headers carry the ranges in real units.

The gridness score is **uncalibrated** — report 2 §2 found random non-negative
weights matching learned gridness (1.21 vs 1.27), so it describes the map and is
not evidence of periodicity.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

N_PER_PAGE = 6
MAX_PAGES = None          # None = every neuron

for variant, out in ratemaps.items():
    n_pages = int(np.ceil(len(out["maps"]) / N_PER_PAGE))
    if MAX_PAGES:
        n_pages = min(n_pages, MAX_PAGES)
    path = os.path.join(SAVE_DIR, f"panels_{variant}.pdf")
    with PdfPages(path) as pdf:
        for p in range(n_pages):
            fig = tm.plot_ratemap_panels(
                out["maps"], out["meta"], out["axes"],
                maps_raw=out["maps_raw"], autocorr=out["autocorr"],
                autocorr_raw=out["autocorr_raw"], page=p,
                n_neurons=N_PER_PAGE, gridness=True)
            pdf.savefig(fig, bbox_inches=None)
            plt.close(fig)
    print(f"{variant}: {n_pages} pages, {len(out['maps'])} neurons -> {path}")

In [ ]:
# Preview inline: first few neurons of each variant.
for variant, out in ratemaps.items():
    fig = tm.plot_ratemap_panels(
        out["maps"], out["meta"], out["axes"], maps_raw=out["maps_raw"],
        autocorr=out["autocorr"], autocorr_raw=out["autocorr_raw"],
        page=0, n_neurons=4, gridness=True)
    plt.show()

### 5b. Compact grids (smoothed only)

The triptych is for looking at individual neurons. These pages fit ~48 neurons
each and are for scanning the population.

In [ ]:
for variant, out in ratemaps.items():
    path = os.path.join(SAVE_DIR, f"ratemaps_{variant}.pdf")
    maps, meta, axes = out["maps"], out["meta"], out["axes"]
    per_page = 48
    n_pages = int(np.ceil(len(maps) / per_page))
    with PdfPages(path) as pdf:
        for p in range(n_pages):
            fig = tm.plot_ratemap_grid(maps, meta, axes, page=p, nrows=6, ncols=8)
            pdf.savefig(fig, bbox_inches=None)
            plt.close(fig)
    print(f"{variant}: {n_pages} pages -> {path}")

## 6. Sorted by peak rate

Native neuron order above. Sorting by peak rate puts the neurons with enough
spikes to be worth looking at first — a *sampling* statement, not a claim that
they are tuned.

In [ ]:
for variant, out in ratemaps.items():
    order = np.argsort(-out["meta"]["peak_hz"].values)
    fig = tm.plot_ratemap_panels(
        out["maps"][order], out["meta"].iloc[order].reset_index(drop=True),
        out["axes"], maps_raw=out["maps_raw"][order],
        autocorr=out["autocorr"][order], autocorr_raw=out["autocorr_raw"][order],
        page=0, n_neurons=6, gridness=True)
    if SAVE_FIGS:
        fig.savefig(os.path.join(SAVE_DIR, f"top_by_rate_{variant}.pdf"),
                    bbox_inches="tight")
    plt.show()